# Correlated noise: common modes

Detectors observed at the same times can share noise, e.g. a common mode of a
wafer from thermal fluctuations. A {class}`~pysanepic.DetectorGroup` with one
or more {class}`~pysanepic.CommonMode` models it: detector $i$ sees
$U_i\,c(t)$ besides its own noise, and $N^{-1}$ is applied with the Woodbury
identity, one detector at a time.

Here: 4 detectors observed at the same times (different pointings), own 1/f
noise plus a strong red common mode, and three noise models for the same
data: own noise only (the correlation ignored), the mode added to the own
spectrum of each detector (its diagonal part), and the group.

In [1]:
import dataclasses

import healpy as hp
import numpy as np
from pysanepic import CommonMode, DetectorData, DetectorGroup, GLSParameters, OneOverFNoise, Pointing, TabulatedNoise, make_maps

nside, fs, chunk, nchunk = 16, 10.0, 6000, 12
n = chunk * nchunk
rng = np.random.default_rng(3)
sky = hp.smoothing(rng.normal(0, 1e-4, (3, 12 * nside**2)), fwhm=np.radians(6))
own = OneOverFNoise(net_ukrts=50.0, fknee_hz=0.02, alpha=1.0, fmin_hz=1e-4)
mode = OneOverFNoise(net_ukrts=20.0, fknee_hz=1.0, alpha=2.0, fmin_hz=1e-3)  # red common mode
coupling = np.array([1.0, 0.9, 1.1, 1.05])


def draw(noise):
    """Noise with the spectrum of `noise`, one realization per chunk."""
    w = noise.weights(chunk, fs)
    return np.concatenate([np.fft.irfft(np.fft.rfft(rng.normal(size=chunk)) / np.sqrt(w), n=chunk) for _ in range(nchunk)])


def scan(n):
    t = np.arange(n)
    lam = 2 * np.pi * t / n + rng.uniform(0, 2 * np.pi)
    spin = 2 * np.pi * t / 3000.0
    a = np.array([np.cos(lam), np.sin(lam), np.zeros(n)])
    z = np.array([np.zeros(n), np.zeros(n), np.ones(n)])
    e2 = np.cross(a.T, z.T).T
    b = np.cos(1.3) * a + np.sin(1.3) * (np.cos(spin) * z + np.sin(spin) * e2)
    return np.arccos(np.clip(b[2], -1, 1)), np.arctan2(b[1], b[0]) % (2 * np.pi), (spin + lam) % (2 * np.pi)


c = draw(mode)
dets = []
for i, pa in enumerate((0.0, np.pi / 4, np.pi / 8, 3 * np.pi / 8)):
    th, ph, psi = scan(n)
    a, p = psi + pa, hp.ang2pix(nside, th, ph)
    tod = sky[0, p] + sky[1, p] * np.cos(2 * a) + sky[2, p] * np.sin(2 * a) + draw(own) + coupling[i] * c
    dets.append(DetectorData(tod=tod, pointing=Pointing(theta=th, phi=ph, psi=psi, coordinates="E"),
                             sampling_rate_hz=fs, noise=own, pol_angle_rad=pa, name=f"d{i}"))

In [2]:
f = np.geomspace(fs / chunk, fs / 2, 200)


def psd(noise):
    s = noise.net_ukrts * np.sqrt(fs) * 1e-6
    return 2 * s**2 / fs * (f**noise.alpha + noise.fknee_hz**noise.alpha) / (f**noise.alpha + noise.fmin_hz**noise.alpha)


runs = {
    "own noise only": dets,
    "own + mode on the diagonal": [dataclasses.replace(d, noise=TabulatedNoise(freqs_hz=f, psd=psd(own) + u**2 * psd(mode)))
                                   for d, u in zip(dets, coupling)],
    "DetectorGroup with the mode": [DetectorGroup(detectors=dets, modes=[CommonMode(coupling=coupling, noise=mode)])],
}
params = GLSParameters(chunk_s=chunk / fs, tol=1e-10)
for name, data in runs.items():
    r = make_maps(data, nside, coordinates="E", params=params)
    seen = r.maps[1] != hp.UNSEEN
    err = (r.maps - sky)[:, seen] * 1e6
    print(f"{name:28s} {r.iterations:3d} iterations, rms error I/Q/U {err[0].std():6.1f} / {err[1].std():6.1f} / {err[2].std():6.1f} uK")

own noise only                26 iterations, rms error I/Q/U  423.6 /  437.9 /  461.3 uK


own + mode on the diagonal   203 iterations, rms error I/Q/U  150.8 /  149.9 /  144.2 uK


DetectorGroup with the mode  150 iterations, rms error I/Q/U   60.2 /   65.2 /   64.8 uK


Modelling the common mode reduces the map error by ~2.5 times with respect
to adding it to the own spectrum of each detector, and by ~7 times with
respect to ignoring it. The PCG needs more iterations than with uncorrelated
noise, because the preconditioner uses only the own noise of each detector.

## Two-level preconditioner

The extra iterations come from a few large-scale modes of I that a
preconditioner local to each pixel cannot fix. With `coarse_nside` the PCG
also solves exactly for the I map at that resolution, at a setup cost of about
one iteration per coarse pixel (12 at `coarse_nside=1`).

In [3]:
group = runs["DetectorGroup with the mode"]
ref = make_maps(group, nside, coordinates="E", params=params)
for nc in (0, 1, 2):
    r = make_maps(group, nside, coordinates="E", params=dataclasses.replace(params, coarse_nside=nc))
    seen = r.maps[1] != hp.UNSEEN
    diff = np.abs(r.maps - ref.maps)[:, seen].max() / np.abs(ref.maps[:, seen]).max()
    print(f"coarse_nside={nc}: {r.iterations:3d} iterations, max difference from coarse_nside=0: {diff:.1e}")

coarse_nside=0: 150 iterations, max difference from coarse_nside=0: 0.0e+00


coarse_nside=1:  85 iterations, max difference from coarse_nside=0: 1.9e-04


coarse_nside=2:  87 iterations, max difference from coarse_nside=0: 2.2e-04


The maps agree within the PCG tolerance. On G100 (16 days, 8 detectors,
nside 512, half of the noise variance in a common mode) `coarse_nside=1` cut
the iterations from 46 to 21 and the GLS time by 33%, setup included; higher
resolutions only add setup. Without common modes the setup costs more than
the iterations it saves, so the default is 0.